<a href="https://colab.research.google.com/github/Cosmosmk2/FashionMNIST/blob/main/FashionMNIST_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import models, transforms
import torchvision.datasets as datasets
from torch.utils.data import Subset, random_split, DataLoader

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
training_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(0.3),
    transforms.RandomVerticalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])
testing_evaluate_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

In [ ]:
training = datasets.FashionMNIST(root = "data", train=True, download=True, transform=training_transform)
testing_set = datasets.FashionMNIST(root = "data", train = False, download = True, transform = testing_evaluate_transform)

training_split = int(len(training) * 0.8)
eval_split = len(training) - training_split

training_set, evalation_set = random_split(training, [training_split, eval_split])

In [ ]:
trainloader = DataLoader(training_set, batch_size=64, shuffle=True)
evalloader = DataLoader(evalation_set, batch_size=100, shuffle=False)
testloader = DataLoader(testing_set, batch_size=1000, shuffle=False)

In [ ]:
i, m = next(iter(trainloader))

# i contains the batch of images, m contains the labels
# Let's inspect an image and a label from the batch, using a valid index (0 to 63):
print(m[0])

In [ ]:
classes = training.classes
classes

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self, loops, filters, stride, padding, k_size):
        super().__init__()
        # Use nn.ModuleList so PyTorch registers the layers correctly
        self.blocks = nn.ModuleList()

        in_channels = 1
        for i in range(loops):
            self.blocks.append(nn.Conv2d(in_channels = in_channels,
                                    out_channels = filters[i],
                                    kernel_size = k_size[i],
                                    stride = stride[i],
                                    padding = padding[i]))
            self.blocks.append(nn.ReLU())
            self.blocks.append(nn.BatchNorm2d(filters[i]))
            in_channels = filters[i]

        self.fc_layer = nn.Sequential(
            nn.AdaptiveAvgPool2d((2, 2)),
            nn.Flatten(),
            nn.Linear(filters[-1] * 2 * 2, 10)
        )

    def forward(self, x):
        for layer in self.blocks:
            x = layer(x)
        x = self.fc_layer(x)
        return x

In [ ]:
model = FashionCNN(3, [3, 100, 300], [1,1,2], [1,1, 2],[3,3,3]).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0007)

In [ ]:
def Training(loader, model, loss_function, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_index, (entry, label) in enumerate(loader):
        entry = entry.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        output = model(entry)
        loss = loss_function(output, label)
        loss.backward()
        optimizer.step()

        _, predicted = output.max(1)
        total += label.size(0)
        correct += predicted.eq(label).sum().item()
        running_loss += loss.item()

        if batch_index % 100 == 0:
            loss = running_loss / (batch_index + 1)
            accuracy = 100 * correct / total
            print(f"Batch: {batch_index}| Loss: {loss:.4f}| Accuracy: {accuracy:.2f}%")


def Evaluation(loader, model, device):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for batch_index, (entry, label) in enumerate(loader):
            entry = entry.to(device)
            label = label.to(device)

            output = model(entry)
            _, predicted = output.max(1)

            total += label.size(0)
            correct += predicted.eq(label).sum().item()

            if batch_index % 500 == 0:
                accuracy = 100 * correct / total
                print(f"Evaluation: Batch: {batch_index}| Accuracy: {accuracy:.2f}%")

In [ ]:
epoch = 10

for epoch in range(epoch):
    print(f"Epoch: {epoch}")
    Training(trainloader, model, loss_function, optimizer, device)
    Evaluation(evalloader, model, device)